# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr, chi2_contingency

In [2]:
# cargar archivos

orders =pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv') # tu código aquí
catalog =pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv') # tu código aquí
marketing =pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv') # tu código aquí


In [3]:
# explorar datasets
# tu código aquí

def explore_data(df, df_name):
    print(f"========== EXPLORING DATASET: {df_name.upper()} ==========\n")
    
    print("--- 1. INFO ---")
    df.info() # Note: info()  
    
    print("\n--- 2. DESCRIBE ---")
    print(df.describe(include='all'))  
    
    print("\n--- 3. DTYPES ---")
    print(df.dtypes)
    
    print("\n--- 4. NUNIQUE (Unique values per column) ---")
    
    for column in df.columns:
        print(f"{column}: {df[column].nunique()}")
        
    print("\n--- 5. VALUE COUNTS (Frequencies per column including NaNs) ---")
   
    for column in df.columns:
        print(f"\nValue counts for column: '{column}'")
        
        print(df[column].value_counts(dropna=False).head(15)) 
        
    print("\n" + "="*60 + "\n")

# 3. Run the function on all three datasets
explore_data(orders, "Orders")
explore_data(catalog, "Catalog")
explore_data(marketing, "Marketing")


========== EXPLORING DATASET: ORDERS ==========

--- 1. INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB

--- 2. DESCRIBE ---
          id_pedido id_usuario fecha_hora_pedido      pais disposit

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [4]:
# problemas encontrados
#fecha_hora_pedido (Orders) y fecha (Marketing) tienen tipo object (texto).
#En Orders, la columna cantidad tiene un valor mínimo de -2.0 (hay cantidades negativas, lo cual es imposible). Además, monto_total tiene un mínimo de -492.65.
#Problema: Hay valores extremos en cantidad (10,000 y 20,000). Es necesario recalcular el monto_total para asegurar que la fórmula (cantidad * precio_unitario) - descuento coincida con el total cobrado.

#En Orders, tienes 25,100 filas, pero id_pedido solo tiene 25,000 valores únicos. Pedidos como order_16301 aparecen 2 veces. Tienes 100 pedidos duplicados.

#En Orders, la columna pais tiene "Colombia", "Mexico", "Argentina", pero también "mexico", "colombia", "argentina" en minúsculas. Para el sistema, "Mexico" y "mexico" son dos países distintos.
#En Orders la categoría es "Electronica" (sin tilde), pero en Catalog es "Electrónica" (con tilde). Si intentas unir las tablas más adelante, no van a coincidir.
#Tienes valores nulos (NaN) en país, dispositivo, canal de marketing, cantidades y precios.
#
#


In [5]:
print("Iniciando limpieza de datos...\n")

# ==========================================
# 1. ELIMINAR DUPLICADOS
# ==========================================
# Eliminamos filas que sean exactamente iguales en Orders
orders = orders.drop_duplicates()
# Por si acaso, eliminamos si hay un mismo id_pedido repetido
orders = orders.drop_duplicates(subset=['id_pedido']) 
print("✔️ Duplicados eliminados en Orders.")


# ==========================================
# 2. CONVERTIR FECHAS AL FORMATO CORRECTO
# ==========================================
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])
print("✔️ Fechas convertidas a formato datetime.")


# ==========================================
# 3. REVISAR VARIABLES CATEGÓRICAS
# ==========================================
# A. Estandarizar países (poner la primera letra en mayúscula para arreglar "mexico" -> "Mexico")
orders['pais'] = orders['pais'].str.title()
marketing['pais'] = marketing['pais'].str.title()

# B. Estandarizar categorías (Quitar la tilde en "Electrónica" del catálogo para que coincida con Orders)
catalog['categoria_producto'] = catalog['categoria_producto'].str.replace('ó', 'o')

# C. Rellenar canales nulos en Marketing usando el id_campaña (ej. "paid_search_Mexico" -> "paid_search")
# Extraemos el canal del nombre de la campaña
marketing['canal'] = marketing['canal'].fillna(marketing['id_campaña'].apply(lambda x: x.rsplit('_', 1)[0]))

print("✔️ Variables categóricas estandarizadas (países, categorías y canales).")


# ==========================================
# 4. REVISAR VARIABLES NUMÉRICAS (Sin negativos)
# ==========================================
# Llenar nulos en numéricas con 0 temporalmente o eliminarlos (aquí los eliminaremos por ser pocos)
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])

# Filtrar para quedarnos SOLO con cantidades mayores a 0
orders = orders[orders['cantidad'] > 0]

# Filtrar para quedarnos SOLO con precios mayores a 0
orders = orders[orders['precio_unitario'] > 0]
print("✔️ Valores negativos y ceros inválidos eliminados.")


# ==========================================
# 5. VERIFICAR CONSISTENCIA DE MONTOS
# ==========================================
# Recalculamos el monto total para asegurarnos de que no haya errores matemáticos
# Fórmula: (Cantidad * Precio Unitario) - Descuento
monto_calculado = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

# Reemplazamos el monto total con el cálculo correcto para garantizar consistencia
orders['monto_total'] = monto_calculado

# Asegurarnos de que el monto total final no sea negativo (si el descuento fue mayor al precio)
orders = orders[orders['monto_total'] >= 0]
print("✔️ Consistencia de montos verificada y corregida.")


print("\n¡Limpieza terminada con éxito! 🚀")
print("Aplicando limpieza final...\n")

# ==========================================
# 1. ELIMINAR OUTLIERS (Cantidades absurdas)
# ==========================================
# Mantenemos solo los pedidos que tengan una cantidad razonable (por ejemplo, menos de 50 artículos)
# Esto eliminará los errores de 10,000 y 20,000
orders = orders[orders['cantidad'] <= 50]
print("✔️ Valores atípicos extremos en 'cantidad' eliminados.")

# ==========================================
# 2. TRATAR VALORES NULOS EN CATEGORÍAS
# ==========================================
# Rellenar los nulos de país, dispositivo y fuente con "Desconocido" para no perder esas ventas
orders['pais'] = orders['pais'].fillna('Desconocido')
orders['dispositivo'] = orders['dispositivo'].fillna('Desconocido')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('Desconocido')

# Si un pedido no tiene nombre_producto ni categoria_producto, es mejor eliminar esa fila 
# ya que no nos sirve para el análisis de ventas por producto
orders = orders.dropna(subset=['nombre_producto', 'categoria_producto'])

print("✔️ Valores nulos tratados correctamente.")
print("\n¡Tus datasets ahora están 100% limpios y listos para el análisis! 🏆")

Iniciando limpieza de datos...

✔️ Duplicados eliminados en Orders.
✔️ Fechas convertidas a formato datetime.
✔️ Variables categóricas estandarizadas (países, categorías y canales).
✔️ Valores negativos y ceros inválidos eliminados.
✔️ Consistencia de montos verificada y corregida.

¡Limpieza terminada con éxito! 🚀
Aplicando limpieza final...

✔️ Valores atípicos extremos en 'cantidad' eliminados.
✔️ Valores nulos tratados correctamente.

¡Tus datasets ahora están 100% limpios y listos para el análisis! 🏆


In [6]:
def explore_data(df, df_name):
    print(f"========== EXPLORING DATASET: {df_name.upper()} ==========\n")
    
    print("--- 1. INFO ---")
    df.info() # Note: info()  
    
    print("\n--- 2. DESCRIBE ---")
    print(df.describe(include='all'))  
    
    print("\n--- 3. DTYPES ---")
    print(df.dtypes)
    
    print("\n--- 4. NUNIQUE (Unique values per column) ---")
    
    for column in df.columns:
        print(f"{column}: {df[column].nunique()}")
        
    print("\n--- 5. VALUE COUNTS (Frequencies per column including NaNs) ---")
   
    for column in df.columns:
        print(f"\nValue counts for column: '{column}'")
        
        print(df[column].value_counts(dropna=False).head(15)) 
        
    print("\n" + "="*60 + "\n")

# 3. Run the function on all three datasets
explore_data(orders, "Orders")
explore_data(catalog, "Catalog")
explore_data(marketing, "Marketing")

========== EXPLORING DATASET: ORDERS ==========

--- 1. INFO ---
<class 'pandas.core.frame.DataFrame'>
Int64Index: 24906 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24906 non-null  object        
 1   id_usuario          24906 non-null  object        
 2   fecha_hora_pedido   24906 non-null  datetime64[ns]
 3   pais                24906 non-null  object        
 4   dispositivo         24906 non-null  object        
 5   fuente_referencia   24906 non-null  object        
 6   nombre_producto     24906 non-null  object        
 7   categoria_producto  24906 non-null  object        
 8   cantidad            24906 non-null  float64       
 9   precio_unitario     24906 non-null  float64       
 10  monto_descuento     24906 non-null  float64       
 11  monto_total         24906 non-null  float64       
dtypes: datetime64[ns](1), float64(4), obj

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [7]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [8]:
# tu código aquí
# ==========================================
# PREPARACIÓN: Unir Orders y Catalog
# ==========================================
# Unimos las tablas usando el nombre y la categoría del producto
df_ventas = pd.merge(orders, catalog, on=['nombre_producto', 'categoria_producto'], how='left')

# Calculamos el costo total de cada pedido (cantidad comprada * costo de ese producto)
df_ventas['costo_total_producto'] = df_ventas['cantidad'] * df_ventas['costo_unitario']

print("📊 PARTE 1: RENTABILIDAD DEL NEGOCIO 📊\n" + "="*40)

# 1. ¿Cuál es el ingreso total (revenue)?
ingreso_total = df_ventas['monto_total'].sum()

# 2. ¿Cuál es el costo total? (Costo de los productos)
costo_productos = df_ventas['costo_total_producto'].sum()

# 3. ¿Cuánto se ha invertido en marketing?
gasto_marketing = marketing['gasto'].sum()

# Costo Total Real (Productos + Marketing)
costo_total = costo_productos + gasto_marketing

# 4. ¿El negocio es rentable? (Profit / Ganancia)
ganancia_neta = ingreso_total - costo_total
es_rentable = "SÍ 🚀" if ganancia_neta > 0 else "NO ⚠️"

# Imprimir resultados formateados con separadores de miles y 2 decimales
print(f"💰 Ingreso Total (Revenue):      ${ingreso_total:,.2f}")
print(f"📦 Costo de Productos (COGS):    ${costo_productos:,.2f}")
print(f"📢 Inversión en Marketing:       ${gasto_marketing:,.2f}")
print("-" * 40)
print(f"📉 Costo Total (Prod + Mkt):     ${costo_total:,.2f}")
print(f"💵 Rentabilidad (Profit):        ${ganancia_neta:,.2f}")
print(f"¿El negocio es rentable?:        {es_rentable}\n")


print("📈 PARTE 2: COMPORTAMIENTO DE VENTAS 📈\n" + "="*40)

# 1. ¿Cuál es el ticket promedio por orden?
# Como cada fila en tu base limpia es 1 pedido único, es el promedio de monto_total
ticket_promedio = df_ventas['monto_total'].mean()

# 2. ¿Cuál es la cantidad promedio de productos por orden?
cantidad_promedio = df_ventas['cantidad'].mean()

# 3. ¿Cuál es el producto más vendido? (Por cantidad total de unidades vendidas)
productos_vendidos = df_ventas.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
producto_top = productos_vendidos.index[0]
cantidad_top = productos_vendidos.iloc[0]

# 4. ¿Cuánto se ha gastado en marketing por canal?
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

# Imprimir resultados
print(f"🛒 Ticket Promedio por Orden:    ${ticket_promedio:,.2f}")
print(f"📦 Cantidad Promedio por Orden:  {cantidad_promedio:,.2f} unidades")
print(f"🏆 Producto más vendido:         {producto_top} ({cantidad_top:,.0f} unidades)")
print("\n📣 Gasto en Marketing por Canal:")
for canal, gasto in gasto_por_canal.items():
    print(f"   - {canal.capitalize()}: ${gasto:,.2f}")

📊 PARTE 1: RENTABILIDAD DEL NEGOCIO 📊
💰 Ingreso Total (Revenue):      $9,610,019.78
📦 Costo de Productos (COGS):    $3,828,869.01
📢 Inversión en Marketing:       $2,871,843.53
----------------------------------------
📉 Costo Total (Prod + Mkt):     $6,700,712.54
💵 Rentabilidad (Profit):        $2,909,307.24
¿El negocio es rentable?:        SÍ 🚀

📈 PARTE 2: COMPORTAMIENTO DE VENTAS 📈
🛒 Ticket Promedio por Orden:    $385.85
📦 Cantidad Promedio por Orden:  1.50 unidades
🏆 Producto más vendido:         Vacuum-Pro-Black (6,284 unidades)

📣 Gasto en Marketing por Canal:
   - Social: $976,818.37
   - Organic: $972,650.96
   - Paid_search: $922,374.20


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [9]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [10]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [11]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
tu código SQL aquí
'''

totals = pd.read_sql(query_totals, con=engine)
totals

ProgrammingError: (psycopg2.errors.SyntaxError) syntax error at or near "tu"
LINE 2: tu código SQL aquí
        ^

[SQL: 
tu código SQL aquí
]
(Background on this error at: http://sqlalche.me/e/14/f405)

In [ ]:

# PARTE 2: Conversiones
# ======================

query_conversion = '''
tu código SQL aquí
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
tu código SQL aquí
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
tu código SQL aquí
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [ ]:
# tu código aquí



---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive